In [ ]:
import pandas as pd

from os import path
import re
import itertools

from pymodulon.io import load_json_model
from pymodulon.plotting import *

import numpy as np
import plotly.express as px
import seaborn as sns

from tqdm.notebook import tqdm

from collections import defaultdict

from matplotlib.lines import Line2D
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
A_condition = pd.DataFrame(index=ica_data.A.index, columns=ica_data.sample_table.full_name.unique())

for condition in ica_data.sample_table.full_name.unique():
    inds = ica_data.sample_table[ica_data.sample_table.full_name == condition].index
    mean = ica_data.A[inds].mean(axis=1)
    A_condition[condition] = mean

In [ ]:
condition_order = [x for x in ica_data.sample_table.full_name.unique() if 'JU' in x]

In [ ]:
from itertools import combinations

nitrogen_dimas = {}
for cond in tqdm(condition_order):
    combination = 'Ammon vs ' + cond
    nitrogen_dimas[combination] = dima(ica_data, cond, 'JU_Nitrogen:Ammon', threshold=15)

In [ ]:
im_appearance = defaultdict(int)
for key, df_diff in nitrogen_dimas.items():
    for im in df_diff.index:
        im_appearance[im] += 1

dima_ims = pd.Series(im_appearance).sort_values()[pd.Series(im_appearance).sort_values() >= len(condition_order) * .5].index

In [ ]:
ims_of_interest = ['Ribosome Biogenesis Repression', 'GLN3', 'HAP4', 'Cell Cycle Progression', 'IFH1', 'Organic Acid Stress', 'DAL Associated', 'Iron Homeostasis Associated']

In [ ]:
im = "GCN4 Associated"
for ind in ica_data.view_imodulon(im).gene_weight.index:
    print(ind, '\t', ica_data.view_imodulon(im).gene_weight.loc[ind])

In [ ]:
positive_aa = ['Arg', 'His', 'Lys', 'Orn']
negative_aa = ['Asp', 'Glu']
polar_aa = ['Ser', 'Thr', 'Asn', 'Gln', 'Cys', 'Pro']
nonpolar_aa = ['Gly', 'Ala', 'Val', 'Leu', 'Met', 'Ile']
aromatic_aa = ['Phe', 'Tyr', 'Trp']
other_aa = ['Cit']

nucleic_acids = ['Aden', 'Cytos', 'Thym', 'Guan', 'Cytid']

other = ['Ammon', 'GABA', 'Urea']

In [ ]:
samples = ica_data.sample_table[ica_data.sample_table.project == 'JU_Nitrogen'].index
A = ica_data.A[samples]
metadata = ica_data.sample_table.loc[samples]

In [ ]:
def classify_compound(name):
    if any(aa in name for aa in positive_aa):
        return 'Positive AA'
    elif any(aa in name for aa in negative_aa):
        return 'Negative AA'
    elif any(aa in name for aa in polar_aa):
        return 'Polar AA'
    elif any(aa in name for aa in nonpolar_aa):
        return 'Nonpolar AA'
    elif any(aa in name for aa in aromatic_aa):
        return 'Aromatic AA'
    elif any(aa in name for aa in other_aa):
        return 'Other AA'
    elif any(na in name for na in nucleic_acids):
        return 'Nucleotides and Nucleosides'
    elif any(ot in name for ot in other):
        return 'Other'
    else:
        return 'Unknown'

metadata['compound_category'] = metadata.condition.map(classify_compound)


In [ ]:
metadata.BioProject

In [ ]:
'''
The samples list define all sample names for which PCA is done, the labels plot labels the sample type for 
the scatter plot
'''

# Perform PCA
pca = PCA(n_components=4)
principal_components = pca.fit_transform(A.T)

# Create a new DataFrame with the principal components
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2', 'PC3', 'PC4'], index = A.columns)

markers = metadata.apply(lambda x: x.condition, axis=1).values
labels = metadata.apply(lambda x: x.condition, axis=1).values

marker_map = {
    'Positive AA': 'o',
    'Negative AA': 's',
    'Polar AA': '^',
    'Nonpolar AA': 'v',
    'Aromatic AA': 'D',
    'Other AA': 'P',
    'Nucleotides and Nucleosides': '*',
    'Other': 'X',
    'Unknown': 'x'
}

# Use one color per nitrogen source (i.e., per condition)
unique_conditions = metadata['condition'].unique()
color_palette = sns.color_palette("Set2", len(unique_conditions)).as_hex()
color_map = dict(zip(unique_conditions, color_palette))

# Calculate centroids for each condition
centroids = {}
for condition in unique_conditions:
    condition_samples = metadata[metadata['condition'] == condition].index
    centroid_pc1 = df_pca.loc[condition_samples, 'PC1'].mean()
    centroid_pc2 = df_pca.loc[condition_samples, 'PC2'].mean()
    centroid_pc3 = df_pca.loc[condition_samples, 'PC3'].mean()
    centroid_pc4 = df_pca.loc[condition_samples, 'PC4'].mean()
    compound_type = metadata.loc[condition_samples[0], 'compound_category']
    
    centroids[condition] = {
        'PC1': centroid_pc1,
        'PC2': centroid_pc2,
        'PC3': centroid_pc3,
        'PC4': centroid_pc4,
        'compound_type': compound_type
    }

# Set up the scatter plots
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Plot centroids
for condition, centroid_data in centroids.items():
    color = color_map.get(condition, 'gray')
    marker_style = marker_map.get(centroid_data['compound_type'], 'x')
    
    ax.scatter(
        centroid_data['PC1'],
        centroid_data['PC2'],
        alpha=0.7,
        s=200,
        c=color,
        marker=marker_style,
        edgecolors='k'
    )
    
    # Add label next to each centroid
    ax.text(
        centroid_data['PC1'] - 3,  # Offset slightly to the right
        centroid_data['PC2'] + 1,  # Offset slightly up
        condition,
        fontsize=12,
        ha='left',
        va='bottom'
    )

ax.set_xlabel(f'PC1 - Explained Variance: {pca.explained_variance_ratio_[0]:.2%}', fontsize=20)
ax.set_ylabel(f'PC2 - Explained Variance: {pca.explained_variance_ratio_[1]:.2%}', fontsize=20)

##Add in arrows for biplot
loadings = pd.DataFrame(index=A.index)
loadings['PC1'] = pca.components_[0, :]
loadings['PC2'] = pca.components_[1, :]

arrows_to_make = list(loadings['PC1'].abs().sort_values(ascending = False).index[:7]) + list(loadings['PC2'].abs().sort_values(ascending = False).index[:1])
arrows_to_make = sorted(set(arrows_to_make))

# arrow length variable extends arrows to help in interpreting the plot
# text_scale determines how far text labels are from arrow tips (1 = on top of them)
arrow_length = 50
text_scale = 1.1

# the place the arrows start is up to you, so you can move it out of the way
start1 = 0
start2 = 0

# loop through each gene and add its arrow
for g in arrows_to_make:
    # directions in PC1 and 2
    g_weight_pc1 = loadings.loc[g, 'PC1']
    g_weight_pc2 = loadings.loc[g, 'PC2']
    
    # lengthen arrow by a constant factor
    g_weight_pc1 = g_weight_pc1 * arrow_length
    g_weight_pc2 = g_weight_pc2 * arrow_length
    
    # add the arrow
    ax.arrow(start1, start2, g_weight_pc1, g_weight_pc2, head_width = 1, color='black')

    ax.text(start1 + (text_scale * g_weight_pc1)+1,
            start2 + (text_scale * g_weight_pc2), 
            g, size=15)

# Create legend for compound types only
included_types = set()
legend_elements = []
for compound_type in sorted(set(metadata['compound_category'])):
    marker_style = marker_map.get(compound_type, 'x')
    legend_elements.append(plt.Line2D(
        [0], [0],
        marker=marker_style,
        color='black',
        label=compound_type,
        markersize=10,
        linestyle='None',
        markerfacecolor='black',
        alpha=1,
        markeredgecolor='k'
    ))

plt.legend(handles=legend_elements, title='Legend', loc='lower right', ncol=2, fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("Activity PCA Plot For Nitrogen Project", fontsize=24)
plt.tight_layout()

plt.savefig('images/fig4_activity_pca.svg', format = 'svg')

In [ ]:
ims_of_interest = arrows_to_make

g = sns.clustermap(A_condition.loc[ims_of_interest,condition_order].T, cmap='coolwarm', center = 0, col_cluster=True, figsize = (5,8))
new_labels = [label.get_text().split(":")[-1] for label in g.ax_heatmap.get_yticklabels()]
g.ax_heatmap.set_yticklabels(new_labels, rotation=0, fontsize= 12);
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=12);
g.ax_heatmap.set_title("Heatmap of iM activities by Nitrogen Source", fontsize=21, pad=80)
g.cax.yaxis.set_tick_params(labelsize=12);
plt.savefig('images/fig4_heatmap.svg', format = 'svg')

In [ ]:
g = sns.clustermap(A_condition.loc[:,condition_order].T, cmap='coolwarm', center = 0, col_cluster=True, figsize = (20,10))
new_labels = [label.get_text().split(":")[-1] for label in g.ax_heatmap.get_yticklabels()]
g.ax_heatmap.set_yticklabels(new_labels, rotation=0, fontsize= 12);
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=12);
g.ax_heatmap.set_title("Heatmap of iM activities by Nitrogen Source", fontsize=21, pad=80)
g.cax.yaxis.set_tick_params(labelsize=12);

In [ ]:
plot_expression(ica_data, 'YER040W', projects = 'JU_Nitrogen')

In [ ]:
plot_activities(ica_data, 'Iron Homeostasis Associated', projects = 'JU_Nitrogen')

In [ ]:
conditions_to_samples = {}
for condition in condition_order:
    conditions_to_samples[condition] = list(ica_data.sample_table[ica_data.sample_table.full_name == condition].index)

high_activity_conditions = ['JU_Nitrogen:Cys', 'JU_Nitrogen:His']
A_plotting = pd.DataFrame(index = high_activity_conditions + ['other'], columns = ['Activity', 'Sample Activities'])
A_plotting.loc[high_activity_conditions, 'Activity'] = A_condition.loc['Iron Homeostasis Associated', high_activity_conditions]
A_plotting.loc['other', 'Activity'] = A_condition.loc['Iron Homeostasis Associated', [x for x in condition_order if x not in high_activity_conditions]].mean()

for cond in high_activity_conditions:
    A_plotting.loc[cond, 'Sample Activities'] = list(ica_data.A.loc['Iron Homeostasis Associated', conditions_to_samples[cond]].values)

A_plotting.loc['other', 'Sample Activities'] = []
for cond in set(condition_order) - set(high_activity_conditions):
    A_plotting.loc['other', 'Sample Activities'] += list(ica_data.A.loc['Iron Homeostasis Associated', conditions_to_samples[cond]].values)

fig, ax = plt.subplots(1,1, figsize = (4.25, 3))

tick_labels = [x.split(':')[-1] for x in A_plotting.index]

bar_xpos = list(range(len(A_plotting)))
colors = [color_map[tick_labels[0]], color_map[tick_labels[1]], 'tab:blue']

ax.bar(bar_xpos, A_plotting.Activity.values, 
       tick_label = [x.split(':')[-1] for x in A_plotting.index],
      color = [color_map[tick_labels[0]], color_map[tick_labels[1]], 'tab:blue'])

for x in range(len(bar_xpos)):
    sample_values = A_plotting.iloc[x, 1]
    ax.scatter([bar_xpos[x]] * len(sample_values), sample_values, color = 'Black', alpha = .7)

plt.ylabel('Activity', fontsize = 14);
plt.title('Activity of Iron \nHomeostasis iM', fontsize = 21);
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)
plt.tight_layout()

plt.savefig('images/fig4_iron_homeostasis_activities.svg', format = 'svg')

In [ ]:
amino_acids = ['Ala', 'Arg', 'Asn', 'Asp', 'Cys', 'Gln', 'Glu', 'Gly', 'His', 'Ile', 
               'Leu', 'Lys', 'Met', 'Phe', 'Pro', 'Ser', 'Thr', 'Trp', 'Tyr', 'Val']

nucleic_acids = ['Aden', 'Cytos', 'Thym', 'Guan']
aa_samples = []
na_samples = []
for cond, samp in conditions_to_samples.items():
    if cond.split(':')[-1] in amino_acids:
        aa_samples += samp
    elif cond.split(':')[-1] in nucleic_acids:
        na_samples += samp

fig, ax = plt.subplots(1,1, figsize = (4.5, 3.5))

plot_dima(ica_data, aa_samples, na_samples, ax = ax);
plt.ylabel('Nucleotides', fontsize = 14)
plt.yticks(fontsize = 12)
plt.xlabel('Amino Acids', fontsize = 14)
plt.xticks(fontsize = 12)
plt.title("Differential iModulon Activity", fontsize = 18)

legend_circle = Line2D([0], [0], marker='o', color='w', label='Significant Difference \n(fdr < .1)',
                        markerfacecolor='tab:blue', markersize=12),
plt.legend(handles=legend_circle);
plt.savefig('images/fig4_nucl_vs_aa_dima.svg', format = 'svg')

In [ ]:
ammon_samples = conditions_to_samples['JU_Nitrogen:Ammon']
glu_samples = conditions_to_samples['JU_Nitrogen:Glu']
df_diff = dima(ica_data, ammon_samples, glu_samples, threshold=10, fdr=0.01)


norm = matplotlib.colors.TwoSlopeNorm(vmin=df_diff['difference'].min(),
                            vcenter=0,
                            vmax=df_diff['difference'].max())

# Map colors from the palette using the normalized values
cmap = plt.get_cmap('seismic')
colors = [cmap(norm(val)) for val in df_diff['difference']]


fig, ax = plt.subplots(1,1, figsize=(3.5,3))
sns.barplot(df_diff, x=df_diff.index, y='difference', ax=ax, palette=colors)
# plt.title('Differential iModulon Activity Between Starvation Samples at 0h and 12h')
# plt.ylabel('Activity Difference')
ax.tick_params(axis='x', rotation=90)
# plt.savefig('images/starvation_dima_barplot.svg', format = 'svg')
# df_diff

# GCN4 Analysis

In [ ]:
im = 'GCN4 Associated'

In [ ]:
plot_gene_weights_chrom(ica_data, im)

# DAL Analysis

In [ ]:
im2 = "DAL Associated"

In [ ]:
plot_gene_weights_chrom(ica_data, im2)

# iM Comparison

In [ ]:
plot_activities(ica_data, "Lysine Metabolism", projects = 'JU_Nitrogen')

In [ ]:
compare_gene_weights(ica_data, im, im2)

In [ ]:
# List of genes is drawn from two sources
# Secondary source for both lists: https://doi.org/10.1534/genetics.111.133306
# Primary source for both lists: https://doi.org/10.1128/MCB.01084-06


ncr_genes_cannonical = """
YNL142W
YKR039W
YIR032C
YJR152W
YIR028W
YIR027C
YIR021W
YIR029W
YIR031C
YJL172W
YPR035W
YKL103C
YBR208C
YKR034W
YHL016C
YEL063C
YLR299W
YEL060C
YLR142W
YOR348C
YHR029C
YIR030C
YJR148W
YDL215C
YGR125W
YJL110C
YGR121C
YHR037W
YDL210W
YLR157C
YLR160C
YLR158C
YLR155C
YPL154C
YCL025C
YPR138C
YAL062W
YGL227W
YPL111W
YGR019W
YBR128C
"""
ncr_genes_cannonical = ncr_genes_cannonical.split()
ncr_genes_cannonical = [x for x in ncr_genes_cannonical if x in ica_data.X.index]


ncr_genes_expanded = """
YDR090C
YMR088C
YDL171C
YPL265W
YIL167W
YBL049W
YLR257W
YNL129W
YIL164C
YPL054W
YNL101W
YJL212C
YOR052C
YDL169C
YHR138C
YPL123C
YLR149C
YOR374W
YFR053C
YCL064C
YOR375C
YJR011C
YNL141W
YDL198C
YDL170W
YEL009C
YHR018C
YPR194C
YKR093W
YCR024C-A
YNR012W
YGR161C
YOL128C
YGR081C
YIL146C
YIL088C
YEL062W
YDL237W
YDR041W
YDL238C
YLR048W
YJR001W
YGL209W
YGL196W
"""
ncr_genes_expanded = ncr_genes_expanded.split()
ncr_genes_expanded = [x for x in ncr_genes_expanded if x in ica_data.X.index]

In [ ]:
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
from matplotlib_venn import venn2


def plot_ncr_overlap_with_chrom(
    ica_data,
    im2,
    im_genes,
    ncr_canonical,
    ncr_expanded,
    gene_weights,
    promoter_gataa=None
):

    im_set  = set(im_genes)
    can_set = set(ncr_canonical)
    total_ncr_set = set(ncr_canonical) | set(ncr_expanded)

    def classify(g):
        if g in can_set:
            return "canonical"
        elif g in total_ncr_set:
            return "expanded"
        else:
            return "novel"

    # ── Figure layout ─────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 6))
    gs = GridSpec(
        2, 2,
        height_ratios=[10, 1],
        width_ratios=[1, 1.2],
        hspace=0.05
    )
    ax1 = fig.add_subplot(gs[:, 0])   # Venn spans both rows
    ax_main = fig.add_subplot(gs[0, 1])
    ax_chrom = fig.add_subplot(gs[1, 1], sharex=ax_main)

    # ── Panel 1: Venn ─────────────────────────────────────────────
    v = venn2(
        [im_set, total_ncr_set],
        set_labels=("DAL Associated iM", "Total NCR genes"),
        ax=ax1,
        set_colors=("red", "blue"),
        alpha=0.3
    )

    im_and_can = im_set & can_set
    im_and_exp = (im_set & total_ncr_set) - can_set

    if v.get_label_by_id("11"):
        v.get_label_by_id("11").set_text(
            f"{len(im_set & total_ncr_set)} total\n"
            f"({len(im_and_can)} canonical,\n"
            f"{len(im_and_exp)} expanded)"
        )

    ax1.set_title("iModulon vs NCR gene sets", fontsize=11)

    # ── Panel 2: chromosome-aligned weights ───────────────────────
    # Build grouping
    all_genes = ica_data.X.index  # or ica_data.gene_table.index

    gene_classes = pd.Series(index=all_genes, dtype="object")
    
    for g in all_genes:
        if g in can_set:
            gene_classes[g] = "canonical"
        elif g in total_ncr_set:
            gene_classes[g] = "expanded"
        elif g in im_set:
            gene_classes[g] = "novel"
        else:
            gene_classes[g] = "background"   # sub-threshold & non-NCR

    color_map = {
        "canonical": "crimson",
        "expanded": "darkorange",
        "novel": "steelblue",
        "background": "lightgrey"
    }

    kwargs = {
        "groups": gene_classes,
        "colors": color_map
    }

    # Use your existing function
    plot_gene_weights_chrom(ica_data, im2, ax=ax_main, **kwargs)

    ax_main.set_xticks([])
    ax_main.set_title("Gene weights across genome (colored by NCR class)")

    # Legend
    legend_patches = [
        patches.Patch(color="crimson", label=f"Canonical NCR (n={len(im_and_can)})"),
        patches.Patch(color="darkorange", label=f"Expanded NCR"),
        patches.Patch(color="steelblue", label="Novel candidate"),
    ]
    ax_main.legend(handles=legend_patches, loc='lower left', frameon=True, fontsize=9)

    # ── Chromosome track ──────────────────────────────────────────
    ax_chrom.set_ylim(0, 1)
    ax_chrom.axis('off')

    for idx, row in ica_data.chrom.iterrows():
        start = row['location']
        width = row['length']

        rect = patches.FancyBboxPatch(
            (start, 0.3),
            width,
            0.4,
            boxstyle="round4,pad=0.01,rounding_size=0.03",
            linewidth=0.8,
            edgecolor='black',
            facecolor='lightgrey'
        )
        ax_chrom.add_patch(rect)

        ax_chrom.text(start + width/2, 0.15, idx,
                      ha='center', va='top', fontsize=10)

    ax_chrom.text(
        ax_chrom.get_xlim()[0], 0.6,
        "Chromosomes",
        fontsize=11
    )

    plt.tight_layout()

    return fig

fig = plot_ncr_overlap_with_chrom(
    ica_data,
    im2,
    ica_data.view_imodulon(im2).index,
    ncr_genes_cannonical,
    ncr_genes_expanded,
    ica_data.M.loc[ica_data.view_imodulon(im2).index, im2],
    promoter_gataa=None
);


fig.savefig("images/fig_4_ncr_overlap.svg", format="svg", bbox_inches="tight")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── gene sets ─────────────────────────────────────────────────────────────────
im_genes  = ica_data.view_imodulon(im2).index
ncr_set   = set(ncr_genes_cannonical + ncr_genes_expanded)
im_set    = set(im_genes)

# get weights for iModulon genes
weights = ica_data.M.loc[list(im_set & set(ica_data.X.index)), im2]

def is_positive(g):
    return weights.get(g, 0) >= 0

# five categories
valid_im_ncr_pos  = [g for g in im_set if g in ncr_set     and g in ica_data.X.index and is_positive(g)]
valid_im_ncr_neg  = [g for g in im_set if g in ncr_set     and g in ica_data.X.index and not is_positive(g)]
valid_im_only_pos = [g for g in im_set if g not in ncr_set and g in ica_data.X.index and is_positive(g)]
valid_im_only_neg = [g for g in im_set if g not in ncr_set and g in ica_data.X.index and not is_positive(g)]
valid_ncr_only    = [g for g in ncr_set if g not in im_set and g in ica_data.X.index]

all_genes = (valid_im_ncr_pos + valid_im_ncr_neg +
             valid_im_only_pos + valid_im_only_neg +
             valid_ncr_only)

# ── correlation matrix ────────────────────────────────────────────────────────
corr = ica_data.X.loc[all_genes].T.corr(method='spearman')

# X_sub = ica_data.X.loc[all_genes]
# A = ica_data.A

# # ensure same sample order
# A = A[X_sub.columns]


# corr = pd.DataFrame(index=all_genes, columns=A.index)

# for gene in all_genes:
#     for imod in A.index:
#         corr.loc[gene, imod] = X_sub.loc[gene].corr(A.loc[imod], method='spearman')

# corr = corr.astype(float)

# ── colour mapping ────────────────────────────────────────────────────────────
color_map = {
    **{g: "crimson"     for g in valid_im_ncr_pos},   # iM + NCR, positive weight
    **{g: "darkblue"    for g in valid_im_ncr_neg},   # iM + NCR, negative weight
    **{g: "darkorange"  for g in valid_im_only_pos},  # novel, positive weight
    **{g: "steelblue"   for g in valid_im_only_neg},  # novel, negative weight
    **{g: "lightgrey"   for g in valid_ncr_only},     # canonical NCR only
}

gene_colors = pd.Series(
    [color_map[g] for g in all_genes],
    index=all_genes
)

# ── clustermap ────────────────────────────────────────────────────────────────
g = sns.clustermap(
    corr,
    cmap="RdBu_r",
    vmin=-1, vmax=1,
    row_colors=gene_colors,
    col_colors=gene_colors,
    xticklabels=True,
    yticklabels=True,
    figsize=(14, 14),
    dendrogram_ratio=0.1,
    colors_ratio=0.02
)

# ── tick label colours ────────────────────────────────────────────────────────
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(),
                              fontsize=6, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(),
                              fontsize=6, rotation=0)

for label in g.ax_heatmap.get_xticklabels():
    label.set_color(color_map.get(label.get_text(), "grey"))
for label in g.ax_heatmap.get_yticklabels():
    label.set_color(color_map.get(label.get_text(), "grey"))

# ── legend ────────────────────────────────────────────────────────────────────
patches = [
    mpatches.Patch(color="crimson",
                   label=f"NCR iModulon ∩ Canonical NCR, positive weight (n={len(valid_im_ncr_pos)})"),
    mpatches.Patch(color="darkblue",
                   label=f"NCR iModulon ∩ Canonical NCR, negative weight (n={len(valid_im_ncr_neg)})"),
    mpatches.Patch(color="darkorange",
                   label=f"Novel iModulon candidate, positive weight (n={len(valid_im_only_pos)})"),
    mpatches.Patch(color="steelblue",
                   label=f"Novel iModulon candidate, negative weight (n={len(valid_im_only_neg)})"),
    mpatches.Patch(color="lightgrey",
                   label=f"Canonical NCR only (n={len(valid_ncr_only)})")
]
g.ax_heatmap.legend(
    handles=patches,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.25, 1.1),
    loc="upper right"
)

# plt.savefig("ncr_clustermap.pdf", bbox_inches="tight", dpi=300)